This notebook summarizes the metrics results for the leaky and spiking SNN models.

In [11]:
import os
import sys
notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import logging
import json
import dataframe_image as dfi
import pandas as pd
from force_regression.config.dataconfig import DataConfig
import force_regression.utils.functions as fn
import force_regression.evaluation.compile_results as cr
from force_regression.models.snn import *
from configs.constants import *
print(f'Notebook dir: {notebook_dir}\nProject dir: {project_dir}.')

logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

Notebook dir: /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding/notebooks/analysis
Project dir: /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
config_path = os.path.join(project_dir, 'configs/config.json')

try:
    with open(config_path, 'r') as config_file:
        config = json.load(config_file)
        root_dir = os.path.join(project_dir, config['root_dir'])
        subject_mappings = config['subject_mappings']
        root_results_dir = os.path.join(project_dir, config['root_results_dir'])
        config['root_dir'] = root_dir
        config['root_results_dir'] = root_results_dir
        print(f'Root directory from config: {root_dir}')
except FileNotFoundError:
    print(f"Error: 'config.json' not found in {config_path}")
except json.JSONDecodeError:
    print("Error: 'config.json' is not a valid JSON file.")
except KeyError:
    print("Error: 'root_dir' not found in 'config.json'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")



Root directory from config: /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding/data/HDiEMG_mu_decomposition


# Data configuration

In [13]:
# used to retrieve the figs_dir
data_config = DataConfig(root_dir=root_dir,
                         root_results_dir = root_results_dir,
                          subject=fn.reverse_remap("S1", subject_mappings), 
                            task_type="Trap", finger_type="Individual", day="Day 1",
                            emg_type="surf", f_samp=10240, subj_map=subject_mappings, segment_hold=True, 
                            verbose=False, common_only=True, images=False, time_to_cut=1,
                            )
output_figures_dir = os.path.join(data_config.root_results_dir, data_config.figs_dir)

snn_spiking_dir = SHALLOW_SPIKING_DOUBLE_FILTER
snn_leaky_dir =  SHALLOW_LEAKY
results_dir = 'output_files'


# Load SNN leaky and spiking results

In [14]:
subjects = [ "S1", "S2"] 

metrics_compiled_df_snn_spiking = cr.load_and_parse_snn_data_to_df(dt=0.01, subject_list=subjects,
                                                                    data_type='metrics_df_cv_snn',
                                                                    network_topology=f'{snn_spiking_dir}',
                                                                    results_dir=results_dir,
                                                                    config=config)

metrics_compiled_df_snn_leaky = cr.load_and_parse_snn_data_to_df(dt=0.01, subject_list=subjects,
                                                                    data_type='metrics_df_cv_snn',
                                                                    network_topology=f'{snn_leaky_dir}',
                                                                    results_dir=results_dir,
                                                                    config=config)


Loading data for subject S1 from /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding/results/output_files/S1/SNN/shallow_spiking_double_filter
Loading file: metrics_df_cv_snn_ep_100_seed_90_taufilt1_0.05_taufilt2_0.08_tausyn_0.01_minit_0.03_dt_0.01_sign_1_mvc_15_kfold_2_hold_False.pkl
Loading data for subject S2 from /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding/results/output_files/S2/SNN/shallow_spiking_double_filter
Loading file: metrics_df_cv_snn_ep_100_seed_90_taufilt1_0.05_taufilt2_0.08_tausyn_0.01_minit_0.03_dt_0.01_sign_1_mvc_15_kfold_2_hold_False.pkl
Loading data for subject S1 from /Users/farahbaracat/Documents/PhD Research/Code/snn-fingerforce-decoding/results/output_files/S1/SNN/shallow_leaky
Loading file: metrics_df_cv_snn_ep_100_seed_90_taufilt1_0.08_tausyn_0.01_taumem_0.08_dt_0.01_sign_1_mvc_15_kfold_2_hold_False.pkl
Loading file: metrics_df_cv_snn_ep_100_seed_90_taufilt1_0.08_tausyn_0.01_taumem_0.08_dt_0.01_sign_-1_mvc_15_

# Get mean metrics across finger tasks

In [16]:
mean_std_df_snn = pd.DataFrame()
mean_std_df_conv = pd.DataFrame()
temp_snn_leaky_df = cr.compute_mean_sd_snn_metric(metrics_compiled_df_snn_leaky,
                                                  mvc=15,
                                                  return_samples=False,
                                                  aux_cols=None)
temp_snn_spiking_df = cr.compute_mean_sd_snn_metric(metrics_compiled_df_snn_spiking,
                                                  mvc=15,
                                                  return_samples=False,
                                                  aux_cols=None)

# add topology column to the SNN dataframes
temp_snn_spiking_df[TOPOLOGY_COL] = SHALLOW_SPIKING_DOUBLE_FILTER
temp_snn_leaky_df[TOPOLOGY_COL] = SHALLOW_LEAKY
mean_std_df_snn = pd.concat([mean_std_df_snn, temp_snn_spiking_df, temp_snn_leaky_df], axis=0, ignore_index=True)


Computing mean across 1 torch seeds:[90]
Computing mean across 1 torch seeds:[90]


In [17]:
# mask based on the network parameters since both topologies have different masks
metrics_list = [RMSE_TEST, R2_TEST, MAE_TEST]
metrics_list = [f"{metric}_mean" for metric in metrics_list] + [f"{metric}_std" for metric in metrics_list]
topology_list = [SHALLOW_LEAKY, SHALLOW_SPIKING_DOUBLE_FILTER]
for topology in topology_list:
    topology_mean_metrics = mean_std_df_snn[mean_std_df_snn[TOPOLOGY_COL]==topology]
    print(f"Topology: {topology} | {topology_mean_metrics.shape}")

    # save df to png the table summary of the metrics. Note that this is the mean across folds, active fingers and directions
    df_styled_snn = topology_mean_metrics.groupby([SUBJECT_COL])[metrics_list].mean().style.background_gradient(cmap='Blues')
    df_styled_snn_per_direction = topology_mean_metrics.groupby([SUBJECT_COL, SIGN_MVC_COL])[metrics_list].mean().style.background_gradient(cmap='Blues')
    
    file_prefix = f'{topology}_average_metrics_table_mvc_15'

    # uncomment to save the results
    # dfi.export(df_styled_snn, os.path.join(output_figures_dir, f'{file_prefix}.png'))
    # dfi.export(df_styled_snn_per_direction, os.path.join(output_figures_dir, f'{file_prefix}_per_direction.png'))


Topology: shallow_leaky | (20, 19)
Topology: shallow_spiking_double_filter | (10, 19)


In [18]:
df_styled_snn

,RMSE_test_mean,R2_test_mean,MAE_test_mean,RMSE_test_std,R2_test_std,MAE_test_std
subject,,,,,,
S1,1.537538,0.912523,0.983123,0.289793,0.039225,0.253512
S2,2.217391,0.826634,1.477614,0.751473,0.111124,0.706467


In [19]:
df_styled_snn_per_direction

,,RMSE_test_mean,R2_test_mean,MAE_test_mean,RMSE_test_std,R2_test_std,MAE_test_std
subject,sign_mvc,,,,,,
S1,1,1.537538,0.912523,0.983123,0.289793,0.039225,0.253512
S2,1,2.217391,0.826634,1.477614,0.751473,0.111124,0.706467
